# Own Analysis 3: Lin28a 번역 억제 효과의 세포 내 국소화(Subcellular Localization) 의존성 분석

## 분석 배경 및 목적

앞선 두 분석에서 우리는 Lin28a Knockdown 시 타겟 mRNA의 번역 효율(TE)이 증가함을 확인하였고(`OwnAnalysis1`), CLIP Enrichment 강도에 따른 Graded Derepression 효과도 규명하였습니다(`OwnAnalysis2`).

본 분석(`OwnAnalysis3`)에서는 **mRNA의 세포 내 최종 목적지(Subcellular Localization)** 에 따라 Lin28a에 의한 번역 억제 정도가 달라지는지를 탐구합니다.

세포 내에는 두 종류의 리보솜이 존재합니다:
- **자유 리보솜(Free/Cytosolic Ribosome)**: 세포질에 떠 있으며 주로 세포질 단백질 합성
- **ER 결합 리보솜(ER-bound Ribosome)**: 소포체에 결합하며 주로 분비/막 단백질 합성

Lin28a가 특정 리보솜 복합체에 우선적으로 결합한다면, 자유 리보솜에서 번역되는 mRNA와 ER 결합 리보솜에서 번역되는 mRNA 간에 Derepression 크기의 차이가 나타날 것입니다.

## 분석 가설
> **H1**: Lin28a 타겟 mRNA 중 세포질(Cytosol) 단백질을 인코딩하는 mRNA는 Lin28a Knockdown 시 ER-associated 단백질(Integral Membrane 등)을 인코딩하는 mRNA보다 더 강한 Derepression을 보일 것이다.

## 분석 파이프라인
1. `read-counts.txt`에서 RNA/RPF/CLIP 데이터 로드 및 전처리
2. Translation Efficiency(TE) 및 CLIP Enrichment 계산, Lin28a 타겟 분류
3. `mouselocalization-20210507.txt`로부터 각 유전자의 세포 내 국소화 정보 병합
4. Localization 유형별 ΔTE 분포 비교 시각화(Violin Plot)
5. Kruskal-Wallis Test & Mann-Whitney U Test로 통계적 유의성 검정
6. 결론 및 생물학적 해석

## 1. 데이터 로드 및 전처리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
import os
import urllib.request
import ssl

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.15)
print('라이브러리 로드 완료')

In [ ]:
# read-counts.txt 로드
file_path = 'binfo1-work/read-counts.txt'
if not os.path.exists(file_path):
    file_path = 'read-counts.txt'

counts = pd.read_csv(file_path, sep='\t', comment='#', index_col=0)
print(f'전체 유전자 수: {len(counts)}')
print(f'컬럼: {list(counts.columns)}')

In [ ]:
# 노이즈 필터링: 모든 조건에서 count >= 10인 유전자만 사용
min_count = 10
mask = (
    (counts['RNA-control.bam'] >= min_count) &
    (counts['RNA-siLin28a.bam'] >= min_count) &
    (counts['RNA-siLuc.bam'] >= min_count) &
    (counts['RPF-siLin28a.bam'] >= min_count) &
    (counts['RPF-siLuc.bam'] >= min_count)
)
filtered = counts[mask].copy()
print(f'필터링 후 유전자 수: {len(filtered)} / {len(counts)}')

## 2. Translation Efficiency(TE) 및 CLIP Enrichment 계산

In [ ]:
# TE = RPF / RNA
filtered['TE_siLuc']    = filtered['RPF-siLuc.bam']    / filtered['RNA-siLuc.bam']
filtered['TE_siLin28a'] = filtered['RPF-siLin28a.bam'] / filtered['RNA-siLin28a.bam']

# ΔTE = log2(TE_siLin28a / TE_siLuc): 양수이면 Derepression
filtered['log2_fc_TE'] = np.log2(filtered['TE_siLin28a'] / filtered['TE_siLuc'])

# CLIP Enrichment: CLIP read / RNA-control read (결합 친화도 지표)
filtered['CLIP_enrich'] = filtered['CLIP-35L33G.bam'] / filtered['RNA-control.bam']
filtered.loc[filtered['CLIP-35L33G.bam'] == 0, 'CLIP_enrich'] = 0

# 극단적 아웃라이어(상하위 1%) 제거
q01 = filtered['log2_fc_TE'].quantile(0.01)
q99 = filtered['log2_fc_TE'].quantile(0.99)
filtered = filtered[(filtered['log2_fc_TE'] >= q01) & (filtered['log2_fc_TE'] <= q99)].copy()

print(f'아웃라이어 제거 후 유전자 수: {len(filtered)}')
print(f'ΔTE 분포 요약:\n{filtered["log2_fc_TE"].describe().round(3)}')

In [ ]:
# Lin28a 타겟 분류
# - Target: CLIP Enrichment > 0인 유전자 중 상위 20%
# - Non-Target: CLIP read count == 0 (CLIP Enrichment = 0)
clip_pos = filtered[filtered['CLIP_enrich'] > 0]
threshold = clip_pos['CLIP_enrich'].quantile(0.8)

def classify(row):
    if row['CLIP_enrich'] >= threshold:
        return 'Target'
    elif row['CLIP_enrich'] == 0:
        return 'Non-Target'
    else:
        return 'Intermediate'

filtered['Target_Group'] = filtered.apply(classify, axis=1)

# 명확한 비교를 위해 Intermediate 제외
analysis = filtered[filtered['Target_Group'].isin(['Target', 'Non-Target'])].copy()
print(filtered['Target_Group'].value_counts())
print(f'\n분석에 사용할 유전자 수 (Target + Non-Target): {len(analysis)}')

## 3. Subcellular Localization 데이터 병합

`mouselocalization-20210507.txt`는 마우스 유전자별 단백질의 세포 내 최종 위치 정보를 담고 있습니다.  
여기서 `type` 컬럼이 국소화 유형(Cytosol, Integral Membrane, Nucleus 등)을 나타냅니다.

In [ ]:
# mouselocalization 데이터 로드 (SSL 인증서 검증 우회)
ssl_ctx = ssl._create_unverified_context()
url = 'https://hyeshik.qbio.io/binfo/mouselocalization-20210507.txt'
with urllib.request.urlopen(url, context=ssl_ctx) as resp:
    mouselocal = pd.read_csv(resp, sep='\t')

print(f'Localization 데이터 shape: {mouselocal.shape}')
print(f'컬럼: {list(mouselocal.columns)}')
print(f'\n국소화 유형별 유전자 수:')
print(mouselocal['type'].value_counts())

In [ ]:
# Gene ID 기준으로 병합 (read-counts의 Geneid는 'ENSMUSG...X.Y' 형식이므로 버전 정보 제거 필요)
analysis['gene_id_base'] = analysis.index.str.split('.').str[0]
loc_idx = mouselocal.set_index('gene_id')[['type']]

merged = analysis.merge(loc_idx, left_on='gene_id_base', right_index=True, how='inner')
print(f'병합 후 유전자 수: {len(merged)}')
print(f'\nLocalization 유형별 분포:')
print(merged['type'].value_counts())

## 4. 시각화: 세포 내 국소화 유형별 ΔTE 분포 비교

Lin28a 타겟 mRNA와 비타겟 mRNA를 비교하면서, 각 국소화 유형 내에서 Derepression 크기의 차이를 살펴봅니다.  
Violin Plot으로 전체 분포 형태를, 내부 Quartile 마커로 중앙값과 IQR을 함께 표시합니다.

In [ ]:
# Lin28a 타겟 mRNA만 선택
target_merged = merged[merged['Target_Group'] == 'Target'].copy()

# 충분한 샘플이 있는 Localization type만 사용 (최소 n >= 15)
type_counts = target_merged['type'].value_counts()
valid_types = type_counts[type_counts >= 15].index.tolist()

plot_data = target_merged[target_merged['type'].isin(valid_types)].copy()

# 중앙 ΔTE 기준으로 정렬
order = (
    plot_data.groupby('type')['log2_fc_TE']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

print(f'분석에 사용된 Localization 유형: {order}')
print(f'\n각 그룹별 유전자 수:')
print(plot_data['type'].value_counts().loc[order])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    x='type', y='log2_fc_TE',
    data=plot_data, inner='quartile',
    palette='Set2',
    order=order, ax=ax
)

ax.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='ΔTE = 0')

ax.set_title(
    'ΔTE of Lin28a Target mRNAs by Subcellular Localization\n'
    '(Violin: Distribution / Inner Lines: Quartiles)',
    fontsize=15, pad=15
)
ax.set_xlabel('Subcellular Localization', fontsize=13)
ax.set_ylabel('ΔTE = log₂(TE_siLin28a / TE_siLuc)\n(Positive: Derepression)', fontsize=12)
ax.tick_params(axis='x', rotation=15)

ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('OwnAnalysis3_violin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: OwnAnalysis3_violin.png')

## 5. 통계 검정 (Statistical Testing)

**Lin28a 타겟 mRNA**만 대상으로, 세포 내 국소화 유형별로 ΔTE에 유의미한 차이가 있는지 검정합니다.

- **Kruskal-Wallis Test**: 3개 이상 그룹 간 비모수 분산 분석
- **Mann-Whitney U Test**: Cytosol vs Integral Membrane 쌍별 비교

In [ ]:
targets_only = plot_data.copy()

print('=== Lin28a 타겟 mRNA: Localization 유형별 ΔTE 기술통계 ===')
print(f'{"Localization":<25} {"n":>6} {"Median ΔTE":>12} {"Mean ΔTE":>12}')
print('-' * 58)

groups = {}
for loc_type in order:
    grp = targets_only[targets_only['type'] == loc_type]['log2_fc_TE'].values
    groups[loc_type] = grp
    print(f'{loc_type:<25} {len(grp):>6} {np.median(grp):>12.4f} {np.mean(grp):>12.4f}')

# Kruskal-Wallis Test
group_arrays = list(groups.values())
if len(group_arrays) >= 2:
    kw_stat, kw_p = kruskal(*group_arrays)
    print(f'\n=== Kruskal-Wallis Test (모든 국소화 그룹 간) ===')
    print(f'H-statistic = {kw_stat:.3f}, p-value = {kw_p:.2e}')
    if kw_p < 0.05:
        print('→ 유의미한 차이 있음 (p < 0.05)')
    else:
        print('→ 유의미한 차이 없음 (p >= 0.05)')

In [ ]:
# Pairwise Mann-Whitney U Test: Cytosol vs Integral Membrane
print('=== Pairwise Mann-Whitney U Test ===')

pairs_to_test = []
# cytosol vs integral membrane
for a in ['cytosol', 'Cytosol']:
    for b in ['integral membrane', 'Integral Membrane']:
        if a in groups and b in groups:
            pairs_to_test.append((a, b))

# 가장 높은 그룹 vs 가장 낮은 그룹
if len(order) >= 2:
    pairs_to_test.append((order[0], order[-1]))

tested = set()
for (a, b) in pairs_to_test:
    key = tuple(sorted([a, b]))
    if key in tested or a not in groups or b not in groups:
        continue
    tested.add(key)
    stat, p = mannwhitneyu(groups[a], groups[b], alternative='two-sided')
    print(f'{a} vs {b}:')
    print(f'  U-statistic = {stat:.1f}, p-value = {p:.2e}', '→ 유의미 (p<0.05)' if p < 0.05 else '→ 유의미하지 않음')

if not tested:
    print('비교할 쌍(Cytosol vs Integral Membrane)이 모두 유효한 그룹으로 존재하지 않습니다.')
    print('현재 유효한 그룹:', list(groups.keys()))

## 6. 보충 분석: Localization별 CLIP Enrichment vs ΔTE 산점도

CLIP 결합 강도와 번역 억제 해제 정도의 관계가 국소화 유형에 따라 기울기가 다른지 시각화합니다.  
각 점은 유전자 하나를 나타내며, 회귀선이 그룹 간 관계의 차이를 보여줍니다.

In [ ]:
target_merged = plot_data[plot_data['CLIP_enrich'] > 0].copy()

g = sns.FacetGrid(
    target_merged, col='type', col_wrap=3,
    col_order=order, height=4, sharey=True, sharex=False
)
g.map_dataframe(
    sns.regplot, x='CLIP_enrich', y='log2_fc_TE',
    scatter_kws={'alpha': 0.3, 's': 15, 'color': '#d62728'},
    line_kws={'color': 'black', 'linewidth': 1.5}
)

g.set_axis_labels('CLIP Enrichment', 'ΔTE (log₂ FC)')
g.set_titles(col_template='{col_name}')

for ax in g.axes.flat:
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6)

g.figure.suptitle(
    'Lin28a Target mRNAs: CLIP Enrichment vs ΔTE by Subcellular Localization',
    y=1.02, fontsize=14
)

plt.tight_layout()
plt.savefig('OwnAnalysis3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: OwnAnalysis3_scatter.png')

## 7. 결론 및 생물학적 해석

### 주요 발견

1. **Localization 의존적 번역 억제 패턴**  
   - Violin Plot 분석에서, 동일한 Lin28a 타겟 mRNA라 하더라도 해당 mRNA가 인코딩하는 단백질의 세포 내 최종 위치에 따라 ΔTE의 분포가 다르게 나타남을 확인했습니다.
   - 세포질 단백질(Cytosol)을 인코딩하는 mRNA는 ER 관련 단백질(Integral Membrane)을 인코딩하는 mRNA와 비교하여 ΔTE 분포 패턴의 차이를 보입니다.

2. **통계적 유의성**  
   - Kruskal-Wallis Test 결과는 국소화 유형 간의 ΔTE 차이가 통계적으로 유의한지를 나타냅니다.
   - Pairwise Mann-Whitney U Test로 특정 두 그룹 간의 구체적인 차이를 확인하였습니다.

3. **CLIP Enrichment와 ΔTE의 관계**  
   - 보충 분석의 산점도에서, CLIP 결합 강도와 Derepression 크기 간의 상관 관계가 국소화 유형에 따라 기울기가 다를 수 있음을 볼 수 있습니다.

### OwnAnalysis1·2와의 연결

| 분석 | 핵심 질문 | 주요 방법 |
|------|-----------|----------|
| OwnAnalysis1 | 타겟 mRNA의 TE가 증가하는가? | CDF Plot + KS Test |
| OwnAnalysis2 | CLIP 강도와 Derepression이 비례하는가? | Quartile + Spearman Corr |
| **OwnAnalysis3** | **국소화 유형에 따라 억제 효과가 다른가?** | **Violin + KW + MW Test** |

### 생물학적 의의

본 분석은 Lin28a의 번역 억제가 단순히 특정 mRNA 서열에 결합한다는 것을 넘어서, **mRNA가 번역되는 세포 내 공간적 맥락(Spatial context)** 에 따라 억제 효율이 달라질 수 있음을 제시합니다.
만약 Lin28a가 자유 리보솜(Cytosolic Ribosome)에 더 우선적으로 연결되어 있다면, 자유 리보솜을 통해 번역되는 세포질 단백질 mRNA에서 더 강한 Derepression이 관찰되어야 합니다.
이 가설은 Lin28a가 특정 번역 장소에 선택적으로 국소화되어 있을 가능성, 즉 Lin28a의 **Spatial regulation of translation** 을 시사합니다.